In [1]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

path = r"D:\models\Qwen3-8B"   # or your local folder
tok = AutoTokenizer.from_pretrained(path)
model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.bfloat16, device_map="auto")

print(f"Memory: {model.get_memory_footprint()/1e9:.1f} GB")   # ~16 GB

c:\Users\rahul\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


FileNotFoundError: No such file or directory: D:\models\Qwen3-8B\model-00001-of-00005.safetensors

In [ ]:
W = model.model.layers[0].mlp.down_proj.weight.float().cpu()

def quantize(W, bits, mode="tensor", group=128):
    qmax = 2**(bits-1) - 1
    if mode == "tensor":  scale = W.abs().max() / qmax
    if mode == "channel": scale = W.abs().amax(dim=1, keepdim=True) / qmax
    if mode == "group":
        Wg = W.reshape(-1, group)
        scale = Wg.abs().amax(dim=1, keepdim=True) / qmax
        return (torch.clamp(torch.round(Wg/scale), -qmax-1, qmax) * scale).reshape(W.shape)
    return torch.clamp(torch.round(W/scale), -qmax-1, qmax) * scale

for bits in [8, 4, 3]:
    for mode in ["tensor", "channel", "group"]:
        err = (W - quantize(W, bits, mode)).pow(2).mean()
        print(f"INT{bits} {mode:8s} MSE: {err:.2e}")

In [ ]:
from transformers import BitsAndBytesConfig

cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True)
model4 = AutoModelForCausalLM.from_pretrained(path, quantization_config=cfg, device_map="auto")
print(f"{model4.get_memory_footprint()/1e9:.1f} GB")   # ~5-6 GB

In [ ]:
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

recipe = GPTQModifier(targets="Linear", scheme="W4A16", ignore=["lm_head"])
oneshot(model=path, dataset="open_platypus", recipe=recipe,
        max_seq_length=2048, num_calibration_samples=512,
        output_dir="Qwen3-8B-GPTQ-W4A16")